# Anggota A — A2: Pembersihan V3.1 (Kandidat, belum final)

**Prasyarat:** dataset `DATASET_V3_KANDIDAT_OPERASIONAL.csv` dari V3. Notebook ini mempertahankan `raw_id` dan partisi, menambahkan kolom `text_clean_v31`, dan membuat audit perubahan. **Jangan mengganti V3 final** sebelum kasus rekomendasi artikel ditinjau.


## 1. Unggah dataset V3
Unggah CSV asli V3. Tidak diperlukan RAW XLSX untuk perbaikan inkremental ini.

In [ ]:
from google.colab import files
from pathlib import Path
up=files.upload()
DATA_PATH=Path('DATASET_V3_KANDIDAT_OPERASIONAL.csv')
assert DATA_PATH.exists(), 'Unggah DATASET_V3_KANDIDAT_OPERASIONAL.csv'
print('Dataset:', DATA_PATH, 'Ukuran MB:',round(DATA_PATH.stat().st_size/1e6,1))


Saving DATASET_V3_KANDIDAT_OPERASIONAL.csv to DATASET_V3_KANDIDAT_OPERASIONAL.csv
Dataset: DATASET_V3_KANDIDAT_OPERASIONAL.csv Ukuran MB: 120.5


## 2. Jalankan pipeline V3.1
Aturan ini **hanya** menghapus frasa navigasi dan penanda dateline/kredit foto yang teridentifikasi. Pada data hasil scraping yang telah digabung menjadi satu baris, judul berita rekomendasi sering melekat pada kalimat berikutnya. Oleh sebab itu judul rekomendasi **tidak dihapus otomatis**; baris terkait ditandai untuk tinjauan lanjut.

In [ ]:
"""A2: Conservative V3 -> V3.1 cleaning with unchanged raw_id / partitions.
Usage: python build_v31.py --input DATASET_V3_KANDIDAT_OPERASIONAL.csv --output HASIL_V31
Do not use output as final data without checking REVIEW_NEEDED.csv and tokenizer stats.
"""
import argparse,csv,json,re,statistics,collections,unicodedata
from pathlib import Path

NAV = re.compile(r'(?i)(?<!\w)(?:lihat\s+juga|baca\s+juga)\s*:\s*')
DATELINE = re.compile(r'(?i)(?<!\w)kompas\s*\.\s*com\s*[-–—]\s*')
PHOTO = re.compile(r'(?i)(?<!\w)kompas\s*\.\s*com\s*/\s*[A-Z][A-Z .-]{4,60}(?=\s|$)')
WORD = re.compile(r"(?u)\b[\w]+(?:[-’'][\w]+)*\b")
CSV_COLS = ['raw_id','source','label','year','title','url','text_raw','text_clean','extract_mode','word_count','label_prefix','decision','reasons','audit_notes','random_split','temporal_split','temporal_main']

def clean_text_v31(text, source):
    """Only remove clearly recognizable formatting markers, not arbitrary media names.
    Returns (cleaned_text, matched marker counts, warnings).
    """
    s = unicodedata.normalize('NFKC',text or '').replace('\u00a0',' ')
    hits=collections.Counter(); warnings=[]
    def del_photo(m): hits['PHOTO_CREDIT']+=1; return ' '
    # Photo credit deletion conservative; review sampled changes
    if source == 'Kompas': s = PHOTO.sub(del_photo,s)
    def strip_nav(m):
        hits['NAV_MARKER']+=1
        # Whole linked headline cannot safely be identified in flattened V3 text.
        # Remove marker ONLY; preserve adjacent words; mark for review.
        warnings.append('RELATED_HEADLINE_MAY_REMAIN')
        return ' '
    s = NAV.sub(strip_nav,s)
    def strip_dateline(m):
        hits['KOMPAS_DATELINE']+=1
        return ' '
    # Remove mechanically only for Kompas original items; for hoax quotations preserve.
    if source == 'Kompas':s=DATELINE.sub(strip_dateline,s)
    elif DATELINE.search(s): warnings.append('KOMPAS_DATELINE_IN_OTHER_SOURCE_REVIEW')
    s=re.sub(r'\s+',' ',s).strip()
    return s,hits,sorted(set(warnings))

def run(input_csv, outdir):
    out=Path(outdir);out.mkdir(parents=True,exist_ok=True)
    per_src=collections.Counter(); labels=collections.Counter();changed=collections.Counter()
    markers_before=collections.Counter();markers_after=collections.Counter();below=[];reviews=[];diffs=[]
    word_lengths=collections.defaultdict(list); kept_ids=set(); partition_counts=collections.Counter()
    f_out=open(out/'DATASET_V3_1_KANDIDAT_OPERASIONAL.csv','w',newline='',encoding='utf-8-sig')
    f_review=open(out/'REVIEW_NEEDED_V31.csv','w',newline='',encoding='utf-8-sig')
    f_diff=open(out/'CONTOH_PERUBAHAN_V31.csv','w',newline='',encoding='utf-8-sig')
    writer=csv.DictWriter(f_out,fieldnames=CSV_COLS+['text_clean_v31','word_count_v31','v31_changed','v31_warnings']);writer.writeheader()
    rw=csv.DictWriter(f_review,fieldnames=['raw_id','source','label','warnings','text_before','text_after','manual_decision','manual_notes']);rw.writeheader()
    dw=csv.DictWriter(f_diff,fieldnames=['raw_id','source','matched','before_context','after_context']);dw.writeheader()
    with open(input_csv,encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        needed={'raw_id','source','label','text_clean','random_split','temporal_split','temporal_main'}
        assert needed.issubset(reader.fieldnames),f'Kolom hilang: {needed-set(reader.fieldnames)}'
        for row in reader:
            raw_id=row['raw_id'];assert raw_id not in kept_ids,f'Duplicate raw_id {raw_id}';kept_ids.add(raw_id)
            src=row['source'];label=row['label'];prev=row['text_clean'];new,hits,warnings=clean_text_v31(prev,src)
            nwords=len(WORD.findall(new)); oldwords=len(WORD.findall(prev))
            per_src[src]+=1;labels[label]+=1;word_lengths[label].append(nwords)
            partition_counts[(row['random_split'],label)]+=1
            markers_before['NAV']+=bool(NAV.search(prev));markers_after['NAV']+=bool(NAV.search(new))
            if label=='0':
                markers_before['NAV_VALID']+=bool(NAV.search(prev));markers_after['NAV_VALID']+=bool(NAV.search(new))
                markers_before['VALID_TOTAL']+=1
            markers_before['KOMPAS_DATELINE']+=bool(DATELINE.search(prev))
            markers_after['KOMPAS_DATELINE']+=bool(DATELINE.search(new))
            for h,n in hits.items():changed[h]+=n
            if nwords<8: below.append({'raw_id':raw_id,'source':src,'label':label,'before_words':oldwords,'after_words':nwords,'newly_short':oldwords>=8})
            if prev!=new:
                # Save a bounded set of examples for manual verification.
                if len(diffs)<1000:
                    spans=[m.start() for m in NAV.finditer(prev)] + [m.start() for m in DATELINE.finditer(prev)]
                    pos=spans[0] if spans else 0
                    diffs.append(1);dw.writerow({'raw_id':raw_id,'source':src,'matched':';'.join(hits), 'before_context':prev[max(0,pos-170):pos+280], 'after_context':new[max(0,pos-170):pos+280]})
            if warnings:
                # preserve all ambiguous examples separately; no guessed final decision
                rw.writerow({'raw_id':raw_id,'source':src,'label':label,'warnings':';'.join(warnings),'text_before':prev[:650], 'text_after':new[:650],'manual_decision':'','manual_notes':''})
            row_out={k:row.get(k,'') for k in CSV_COLS}
            row_out.update({'text_clean_v31':new,'word_count_v31':nwords,'v31_changed':int(new!=prev),'v31_warnings':';'.join(warnings)})
            writer.writerow(row_out)
    f_out.close();f_review.close();f_diff.close()
    with open(out/'TEXT_UNDER_8_WORDS_V31.csv','w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=['raw_id','source','label','before_words','after_words','newly_short']);w.writeheader();w.writerows(below)
    n=len(kept_ids)
    summary={'status':'PROVISIONAL_V31_A2','total_rows':n,'by_source':dict(per_src),'by_label':dict(labels),
        'random_partition_counts':{str(a)+'_'+str(b):v for (a,b),v in partition_counts.items()},
        'changed_marker_occurrences':dict(changed),'markers_texts_before':dict(markers_before),'markers_texts_after':dict(markers_after),
        'valid_marker_rate_after_percent':round(100*markers_after['NAV_VALID']/max(1,markers_before['VALID_TOTAL']),4),
        'below_8_words':len(below),'newly_below_8_words':sum(bool(x['newly_short']) for x in below),
        'median_words_by_label':{k:statistics.median(v) for k,v in word_lengths.items()},
        'warning':'Marker removal alone may leave related-article headlines; review flagged rows before calling dataset final. Tokenizer stats pending.'}
    (out/'RINGKASAN_V31.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(summary,ensure_ascii=False,indent=2));return summary
run(DATA_PATH,'HASIL_V31')


{
  "status": "PROVISIONAL_V31_A2",
  "total_rows": 29189,
  "by_source": {
    "CNN Indonesia": 9625,
    "Kompas": 4656,
    "Tempo": 6586,
    "TurnBackHoax": 8322
  },
  "by_label": {
    "0": 20867,
    "1": 8322
  },
  "random_partition_counts": {
    "LATIH_0": 16693,
    "VALIDASI_0": 2087,
    "UJI_0": 2087,
    "LATIH_1": 6658,
    "UJI_1": 832,
    "VALIDASI_1": 832
  },
  "changed_marker_occurrences": {
    "NAV_MARKER": 21227,
    "PHOTO_CREDIT": 286,
    "KOMPAS_DATELINE": 6
  },
  "markers_texts_before": {
    "NAV": 10857,
    "NAV_VALID": 10855,
    "VALID_TOTAL": 20867,
    "KOMPAS_DATELINE": 16
  },
  "markers_texts_after": {
    "NAV": 0,
    "NAV_VALID": 0,
    "KOMPAS_DATELINE": 10
  },
  "valid_marker_rate_after_percent": 0.0,
  "below_8_words": 0,
  "newly_below_8_words": 0,
  "median_words_by_label": {
    "0": 310,
    "1": 31.0
  },
  "warning": "Marker removal alone may leave related-article headlines; review flagged rows before calling dataset final. Tokeni

{'status': 'PROVISIONAL_V31_A2',
 'total_rows': 29189,
 'by_source': {'CNN Indonesia': 9625,
  'Kompas': 4656,
  'Tempo': 6586,
  'TurnBackHoax': 8322},
 'by_label': {'0': 20867, '1': 8322},
 'random_partition_counts': {'LATIH_0': 16693,
  'VALIDASI_0': 2087,
  'UJI_0': 2087,
  'LATIH_1': 6658,
  'UJI_1': 832,
  'VALIDASI_1': 832},
 'changed_marker_occurrences': {'NAV_MARKER': 21227,
  'PHOTO_CREDIT': 286,
  'KOMPAS_DATELINE': 6},
 'markers_texts_before': {'NAV': 10857,
  'NAV_VALID': 10855,
  'VALID_TOTAL': 20867,
  'KOMPAS_DATELINE': 16},
 'markers_texts_after': {'NAV': 0, 'NAV_VALID': 0, 'KOMPAS_DATELINE': 10},
 'valid_marker_rate_after_percent': 0.0,
 'below_8_words': 0,
 'newly_below_8_words': 0,
 'median_words_by_label': {'0': 310, '1': 31.0},
 'warning': 'Marker removal alone may leave related-article headlines; review flagged rows before calling dataset final. Tokenizer stats pending.'}

## 3. Periksa status target dan pembagian
Target formal penanda kurang dari 1% tidak sama dengan seluruh konten rekomendasi sudah bersih. Periksa berkas `REVIEW_NEEDED_V31.csv`.

In [ ]:
from pathlib import Path
import json
s=json.loads(Path('HASIL_V31/RINGKASAN_V31.json').read_text())
assert s['total_rows']==29189
assert s['by_label']=={'0':20867,'1':8322}
print('Sisa penanda navigasi pada VALID (%)',s['valid_marker_rate_after_percent'])
print('Baris turun <8 kata:',s['newly_below_8_words'])
print('Median kata (definisi V3):',s['median_words_by_label'])
print('Peringatan: headline artikel terkait mungkin masih ada; review sebelum final.')


Sisa penanda navigasi pada VALID (%) 0.0
Baris turun <8 kata: 0
Median kata (definisi V3): {'0': 310, '1': 31.0}
Peringatan: headline artikel terkait mungkin masih ada; review sebelum final.


## 4. Statistik token IndoBERT (opsional, diperlukan sebelum laporan final)
Jalankan dengan tokenizer yang sama seperti Tahap 2. Membutuhkan akses internet Colab untuk memuat tokenizer Hugging Face.

In [ ]:
# Uncomment dan jalankan setelah pemeriksaan kebersihan V3.1 selesai
# %pip -q install transformers
# from transformers import AutoTokenizer
# import csv, collections, statistics, json
# tok=AutoTokenizer.from_pretrained('indobenchmark/indobert-base-p1')
# counts=collections.defaultdict(list)
# with open('HASIL_V31/DATASET_V3_1_KANDIDAT_OPERASIONAL.csv',encoding='utf-8-sig',newline='') as f:
#     reader=csv.DictReader(f)
#     for r in reader:
#         counts[r['label']].append(len(tok(r['text_clean_v31'],add_special_tokens=True,truncation=False)['input_ids']))
# summary={k:{'n':len(v),'median_token':statistics.median(v),'over_256':sum(x>256 for x in v),'percent_over_256':100*sum(x>256 for x in v)/len(v)} for k,v in counts.items()}
# print(json.dumps(summary,indent=2))
# Path('HASIL_V31/TOKEN_INDOBERT_V31.json').write_text(json.dumps(summary,indent=2))


## 5. Unduh seluruh keluaran A2
`REVIEW_NEEDED_V31.csv` berisi kasus ambigu dan tidak perlu ditandai seluruhnya pada malam yang sama; mulai dari sampel manual berbasis sumber. Notebook ini tetap menghasilkan file lengkap untuk pelacakan.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files
with zipfile.ZipFile('HASIL_A2_V31_KANDIDAT.zip','w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for p in Path('HASIL_V31').glob('*'):
        if p.is_file():z.write(p,p.name)
files.download('HASIL_A2_V31_KANDIDAT.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>